# Week 11 Walkthrough — JSON In, Database Out

**Topic:** JSON parsing and generation, SQL queries from Python

Two ways to keep data: a JSON file, and a database. We'll go from one to the other and back, running real SQL from Python. We use SQLite because it ships with Python — every query here works the same against your PostgreSQL, with one line changed.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — JSON: a string on one side, Python on the other


`dumps`/`loads` work with strings. `dump`/`load` (no *s*) work with files. That
`s` is the only difference, and it catches everyone once.

In [ ]:
import json

record = {"title": "Dune", "author": "Herbert", "year": 1965, "subjects": ["sf"]}

text = json.dumps(record)
print(type(text), text)

back = json.loads(text)
print(type(back), back["title"])

## Step 2 — JSON to and from a file


`indent=2` makes it readable by a human. Leave it out for compactness.

In [ ]:
import json

catalog = [
    {"title": "Dune", "author": "Herbert", "year": 1965, "loans": 42},
    {"title": "Beloved", "author": "Morrison", "year": 1987, "loans": 93},
    {"title": "Neuromancer", "author": "Gibson", "year": 1984, "loans": 17},
    {"title": "Silent Spring", "author": "Carson", "year": 1962, "loans": 61},
]

with open("catalog.json", "w") as file:
    json.dump(catalog, file, indent=2)

with open("catalog.json") as file:
    loaded = json.load(file)

print(len(loaded), "records reloaded")
print(loaded[1])

## Step 3 — Connect to a database


A connection, then a cursor. The cursor is what you run statements on.

In [ ]:
import sqlite3

connection = sqlite3.connect("library.db")
cursor = connection.cursor()

print(connection)

## Step 4 — Create a table


SQL is its own language. Python just carries the string across.

In [ ]:
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("""
    CREATE TABLE books (
        id      INTEGER PRIMARY KEY,
        title   TEXT NOT NULL,
        author  TEXT NOT NULL,
        year    INTEGER,
        loans   INTEGER DEFAULT 0
    )
""")
connection.commit()
print("table created")

## Step 5 — Insert with placeholders — never with f-strings


The `?` marks are not a style preference. Building SQL by string formatting is
how SQL-injection happens; placeholders let the driver escape values safely.

In [ ]:
rows = [(r["title"], r["author"], r["year"], r["loans"]) for r in catalog]

cursor.executemany(
    "INSERT INTO books (title, author, year, loans) VALUES (?, ?, ?, ?)",
    rows
)
connection.commit()

print(cursor.rowcount, "rows inserted")

## Step 6 — Query it back


`fetchall()` gives every row, `fetchone()` gives the next one. Rows come back as
tuples.

In [ ]:
cursor.execute("SELECT title, author, loans FROM books ORDER BY loans DESC")

for title, author, loans in cursor.fetchall():
    print(f"{loans:3}  {title:16} {author}")

## Step 7 — Filtering and aggregating in SQL


Let the database do the work. It is far better at this than a Python loop over
everything.

In [ ]:
cursor.execute("SELECT COUNT(*), SUM(loans), AVG(loans) FROM books")
count, total, average = cursor.fetchone()
print(f"{count} titles, {total} loans, {average:.1f} average")

cursor.execute("SELECT title, year FROM books WHERE year < ? ORDER BY year", (1980,))
print("\nPublished before 1980:")
for title, year in cursor.fetchall():
    print(f"  {year}  {title}")

## Step 8 — The same code against PostgreSQL


Change the import and the connect line; the cursor API is the same. That is what
the DB-API standard buys you.

In [ ]:
postgres_example = '''
import psycopg2                       # instead of sqlite3

connection = psycopg2.connect(
    host="localhost", dbname="library", user="tbowman", password="..."
)
cursor = connection.cursor()

cursor.execute("SELECT title FROM books WHERE loans > %s", (50,))   # %s, not ?
print(cursor.fetchall())
'''
print(postgres_example)

---

## The finished program

Everything above, in one place. This is the version worth keeping.


JSON file in, database out, query, and a JSON summary back out. Close the
connection when you're finished with it.

In [ ]:
# Week 11 - JSON to SQL and back

import json
import sqlite3

with open("catalog.json") as file:
    catalog = json.load(file)

connection = sqlite3.connect("library.db")
connection.row_factory = sqlite3.Row      # rows behave like dictionaries
cursor = connection.cursor()

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("""
    CREATE TABLE books (
        id     INTEGER PRIMARY KEY,
        title  TEXT NOT NULL,
        author TEXT NOT NULL,
        year   INTEGER,
        loans  INTEGER DEFAULT 0
    )
""")
cursor.executemany(
    "INSERT INTO books (title, author, year, loans) VALUES (:title, :author, :year, :loans)",
    catalog
)
connection.commit()

cursor.execute("SELECT * FROM books WHERE loans >= ? ORDER BY loans DESC", (40,))
popular = [dict(row) for row in cursor.fetchall()]

print("POPULAR TITLES (40+ loans)")
print("=" * 44)
for row in popular:
    print(f"{row['loans']:3}  {row['title']:16} {row['year']}")

cursor.execute("SELECT COUNT(*) AS n, SUM(loans) AS total FROM books")
stats = dict(cursor.fetchone())
print("=" * 44)
print(f"{stats['n']} titles, {stats['total']} loans total")

with open("popular.json", "w") as file:
    json.dump(popular, file, indent=2)
print("\nwrote popular.json")

connection.close()

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Add an `UPDATE books SET loans = loans + 1 WHERE title = ?` statement and re-query to confirm it took.

**2.** Write a query for the average `loans` grouped by decade — `GROUP BY year / 10` gets you started.

**3.** Try building a query with an f-string and a title containing an apostrophe. Watch it break, then fix it with a placeholder.

**4.** Re-run the finished cell twice. Why does `DROP TABLE IF EXISTS` matter?

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3